# GBSED Real Wireless Transmission Evaluation Notebook
### Empirical Evaluation of Actual OMNeT++ / Veins Received Payloads

---

## Overview
This notebook evaluates **actual binary scene graph payloads** received over real/simulated wireless channels from **OMNeT++ / Veins** against the original ground-truth transmitted files from GBSED.

### Core Principles:
1. **No Artificial / Synthetic Noise:** Channel degradation is evaluated purely from real received `.bin` files.
2. **Official GBSED Decoding Pipeline:** Uses exact deserialization (`_format_loading_`), semantic decompression (`sem_decompression`), and graph reconstruction (`sg_autoencoder.decode`).
3. **Multi-Condition Evaluation:** Supports simultaneous benchmarking of multiple actual wireless runs (e.g. **Good**, **Medium**, **Poor** channel conditions from OMNeT++).

### Metrics Calculated per Wireless Output:
- **Packet / File Delivery Status:** Byte count integrity, truncation detection, and decoding success.
- **Empirical Bit Error Rate (BER):** Real bit discrepancy between transmitted and received payloads.
- **Node / Label Accuracy:** Precision of detected road actors and node labels.
- **Feature Matrix MAE & MSE:** Coordinate and distance errors across bounding box and BEV features.
- **Relation Precision, Recall, and F1-Score:** Edge recovery fidelity.
- **Missing & Extra Relations:** Exact identification of lost or hallucinated relational edges.
- **Overall Semantic Fidelity:** End-to-end semantic accuracy score.

## 1. Setup & Imports

In [1]:
import os
import sys
import math
import hashlib
import struct
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure Pandas table display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)

# Handle display in both Jupyter notebook and standalone CLI environments
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)

print("Libraries loaded successfully.")
print(f"NumPy version : {np.__version__}")
print(f"Pandas version: {pd.__version__}")


Libraries loaded successfully.
NumPy version : 2.5.3
Pandas version: 3.0.5


## 2. Configuration & Dataset Paths
Define the paths for the **original sent payload** and the **actual OMNeT++ / Veins output files** for different wireless conditions.

In [2]:
# Determine project root
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

# 1. Original Ground-Truth Sent File
SENT_BIN_PATH = PROJECT_ROOT / "scene_data" / "scene_0000.bin"

# 2. Actual Received Files from OMNeT++ / Veins
# Map condition names to the actual received .bin file paths
ACTUAL_WIRELESS_RUNS = {
    "Good Channel (Actual Veins)": PROJECT_ROOT / "received" / "received_scene_0000.bin",
    "Medium Channel (Actual Veins)": PROJECT_ROOT / "received" / "received_test.bin",
    "Poor Channel (Actual Veins)": PROJECT_ROOT / "received" / "received_poor.bin",  # Update path when generated
}

print("Ground-Truth Sent Path:", SENT_BIN_PATH)
print("Sent File Exists       :", SENT_BIN_PATH.is_file())
print("\nConfigured Actual Wireless Runs:")
for label, path in ACTUAL_WIRELESS_RUNS.items():
    print(f"  - {label:32s}: {path.name} (Exists: {path.is_file()})")

Ground-Truth Sent Path: C:\Users\ACER\Desktop\thesis\gbsed project\gbsed\scene_data\scene_0000.bin
Sent File Exists       : True

Configured Actual Wireless Runs:
  - Good Channel (Actual Veins)     : received_scene_0000.bin (Exists: True)
  - Medium Channel (Actual Veins)   : received_test.bin (Exists: True)
  - Poor Channel (Actual Veins)     : received_poor.bin (Exists: False)


## 3. Official GBSED Schema & Constants
From `Config/pipeline_extraction.yaml`:

In [3]:
ACTOR_NAMES = [
    "ego_car", "car", "moto", "bicycle", "ped", "lane", "light", "sign", "road"
]

RELATION_NAMES = [
    "isIn", "inDFrontOf", "inSFrontOf", "atDRearOf", "atSRearOf", 
    "toLeftOf", "toRightOf", "near_coll", "super_near", "very_near", "near", "visible"
]

FEATURE_KEYS = [
    'left', 'top', 'right', 'bottom', 
    'location_x', 'location_y', 'rel_location_x', 'rel_location_y', 'distance_abs'
]

print(f"Defined Actor Classes    ({len(ACTOR_NAMES)}): {ACTOR_NAMES}")
print(f"Defined Relation Classes ({len(RELATION_NAMES)}): {RELATION_NAMES}")

Defined Actor Classes    (9): ['ego_car', 'car', 'moto', 'bicycle', 'ped', 'lane', 'light', 'sign', 'road']
Defined Relation Classes (12): ['isIn', 'inDFrontOf', 'inSFrontOf', 'atDRearOf', 'atSRearOf', 'toLeftOf', 'toRightOf', 'near_coll', 'super_near', 'very_near', 'near', 'visible']


## 4. Binary Layer Analysis & Empirical BER Calculation
Calculates physical payload delivery, bit error count, and real Bit Error Rate (BER).

In [4]:
def analyze_binary_integrity(path_sent, path_received):
    """
    Performs deep binary comparison between sent and actual received files.
    """
    if not path_sent.is_file():
        raise FileNotFoundError(f"Sent file not found: {path_sent}")
    if not path_received.is_file():
        return {"file_exists": False, "error": "File not found on disk"}
        
    with open(path_sent, "rb") as f:
        sent_bytes = f.read()
    with open(path_received, "rb") as f:
        rec_bytes = f.read()
        
    size_sent = len(sent_bytes)
    size_rec = len(rec_bytes)
    sent_sha = hashlib.sha256(sent_bytes).hexdigest()
    rec_sha = hashlib.sha256(rec_bytes).hexdigest()
    
    # Delivery Status
    if size_rec == size_sent:
        delivery_status = "Delivered (Full Size)"
    elif size_rec == 0:
        delivery_status = "Lost (0 Bytes)"
    elif size_rec < size_sent:
        delivery_status = f"Truncated ({size_rec}/{size_sent} Bytes)"
    else:
        delivery_status = f"Oversized ({size_rec}/{size_sent} Bytes)"
        
    # Bit Error Rate (BER) Calculation
    min_len = min(size_sent, size_rec)
    max_len = max(size_sent, size_rec)
    
    byte_diffs = sum(1 for i in range(min_len) if sent_bytes[i] != rec_bytes[i]) + (max_len - min_len)
    bit_diffs = sum(bin(sent_bytes[i] ^ rec_bytes[i]).count('1') for i in range(min_len))
    bit_diffs += (max_len - min_len) * 8
    
    total_bits = size_sent * 8
    ber = bit_diffs / total_bits if total_bits > 0 else 0.0
    
    return {
        "file_exists": True,
        "size_sent": size_sent,
        "size_rec": size_rec,
        "delivery_status": delivery_status,
        "sent_sha": sent_sha,
        "rec_sha": rec_sha,
        "is_identical": (sent_bytes == rec_bytes),
        "byte_diffs": byte_diffs,
        "bit_diffs": bit_diffs,
        "ber": ber
    }

## 5. Official GBSED Decoding Pipeline
Faithfully implements `pipeline.GBSED._format_loading_`, `sg_autoencoder.sem_decompression`, and `sg_autoencoder.decode` with channel-corruption safety checks.

In [5]:
def format_loading(to_read):
    """
    Official GBSED deserialization from 1D float16 array (pipeline.py lines 130-186).
    """
    cur_idx = 0
    
    # 1. Labels
    if len(to_read) < 1:
        raise ValueError("Empty payload.")
    nb = int(to_read[cur_idx]); cur_idx += 1
    end_idx = cur_idx + nb
    if len(to_read) < end_idx:
        raise ValueError(f"Truncated labels segment (expected {nb} values).")
    labels = to_read[cur_idx : end_idx]
    
    # 2. Features
    cur_idx = end_idx
    nb = int(to_read[cur_idx]); cur_idx += 1
    end_idx = cur_idx + nb
    if len(to_read) < end_idx:
        raise ValueError(f"Truncated features segment (expected {nb} values).")
    features = to_read[cur_idx : end_idx]
    
    # 3. Active relation indices L
    cur_idx = end_idx
    nb = int(to_read[cur_idx]); cur_idx += 1
    end_idx = cur_idx + nb
    if len(to_read) < end_idx:
        raise ValueError(f"Truncated relation indices segment (expected {nb} values).")
    L = to_read[cur_idx : end_idx]
    
    # 4. Compressed tensor comp_T
    cur_idx = end_idx
    nb = int(to_read[cur_idx]); cur_idx += 1
    end_idx = cur_idx + nb
    comp_T = to_read[cur_idx : ]
    
    # Formatting & Reshaping
    labels = [int(i) for i in labels]
    feature_nodes = features.reshape(((len(labels)), -1))
    L = [int(i) for i in L]
    comp_T = comp_T.reshape((len(L), len(labels), len(labels)))
    
    return labels, feature_nodes, L, comp_T

def sem_decompression(comp_T, L, num_relations=len(RELATION_NAMES)):
    """
    Official GBSED inverse semantic compression (sg_autoencoder.py lines 111-134).
    """
    shape = comp_T.shape[1:]
    T = np.zeros((num_relations, shape[0], shape[1]), dtype=comp_T.dtype)
    for i in range(num_relations):
        if i in L:
            pos = L.index(i)
            T[i] = comp_T[pos].copy()
    return T

def decode_gbsed_scenegraph(labels, feature_nodes, L, comp_T):
    """
    Reconstructs node names, attributes, and directed relation edges.
    Corresponds directly to sg_autoencoder.decode() (lines 185-203).
    """
    T = sem_decompression(comp_T, L)
    
    # 1. Recover node names (sg_autoencoder._get_all_node_names)
    node_names = []
    for i, l_idx in enumerate(labels):
        if 0 <= l_idx < len(ACTOR_NAMES):
            name = ACTOR_NAMES[l_idx]
        else:
            name = f"unknown_actor_{l_idx}"
            
        if name == "ego_car":
            to_append = "ego car"
        elif name == "road":
            to_append = "Root Road"
        elif name == "lane":
            if "Right Lane" in node_names:
                to_append = "Middle Lane"
            else:
                to_append = "Right Lane" if "Left Lane" in node_names else "Left Lane"
        else:
            suffix = int(feature_nodes[i, -1]) if feature_nodes.shape[1] > 0 else i
            to_append = f"{name}_{suffix}"
        node_names.append(to_append)
        
    # 2. Recover node attributes (sg_autoencoder._get_node)
    nodes_data = []
    for i, name in enumerate(node_names):
        l_idx = labels[i]
        actor_type = ACTOR_NAMES[l_idx] if 0 <= l_idx < len(ACTOR_NAMES) else "unknown"
        attr = {}
        if "_" in name:
            for k_idx, k in enumerate(FEATURE_KEYS):
                if k_idx < feature_nodes.shape[1]:
                    attr[k] = float(feature_nodes[i, k_idx])
        elif "road" in name.lower():
            attr['name'] = "Root Road"
        elif "ego" in name.lower():
            attr['location_x'] = float(feature_nodes[i, 0]) if feature_nodes.shape[1] > 0 else 0.0
            attr['location_y'] = float(feature_nodes[i, 1]) if feature_nodes.shape[1] > 1 else 0.0
        nodes_data.append({
            "name": name,
            "label": actor_type,
            "value": l_idx,
            "attr": attr
        })
        
    # 3. Recover directed relation edges
    edges = []
    num_nodes = len(node_names)
    for r in range(len(RELATION_NAMES)):
        for j in range(num_nodes):
            for k in range(num_nodes):
                if T[r, j, k] != 0:
                    edges.append((node_names[j], RELATION_NAMES[r], node_names[k]))
                    
    return {
        "labels": labels,
        "features": feature_nodes,
        "L": L,
        "comp_T": comp_T,
        "tensor_T": T,
        "node_names": node_names,
        "nodes": nodes_data,
        "edges": edges
    }

def decode_bin_file(bin_path):
    """Loads a .bin file and decodes it completely into a structured graph dict."""
    with open(bin_path, "rb") as f:
        raw_bytes = f.read()
    float_array = np.frombuffer(raw_bytes, dtype=np.float16)
    labels, feat_nodes, L, comp_T = format_loading(float_array)
    return decode_gbsed_scenegraph(labels, feat_nodes, L, comp_T)

## 6. SceneGraph Comparison & Metric Engine

In [6]:
def compare_scenegraphs(orig, rec):
    """
    Compares original vs received SceneGraphs across nodes, feature matrices, and relations.
    """
    # 1. Node & Label Evaluation
    orig_nodes = orig["node_names"]
    rec_nodes = rec["node_names"]
    orig_labels = orig["labels"]
    rec_labels = rec["labels"]
    
    min_nodes = min(len(orig_labels), len(rec_labels))
    max_nodes = max(len(orig_labels), len(rec_labels))
    matched_labels = sum(1 for i in range(min_nodes) if orig_labels[i] == rec_labels[i])
    node_acc = matched_labels / max_nodes if max_nodes > 0 else 1.0
    
    # 2. Feature Matrix Evaluation (MAE / MSE)
    orig_feat = orig["features"]
    rec_feat = rec["features"]
    
    if orig_feat.shape == rec_feat.shape:
        feat_diff = np.abs(orig_feat.astype(np.float64) - rec_feat.astype(np.float64))
        feat_mae = float(np.mean(feat_diff))
        feat_mse = float(np.mean(feat_diff ** 2))
    else:
        # Handle node dimension mismatch due to severe channel corruption
        min_r = min(orig_feat.shape[0], rec_feat.shape[0])
        min_c = min(orig_feat.shape[1], rec_feat.shape[1])
        if min_r > 0 and min_c > 0:
            diff = np.abs(orig_feat[:min_r, :min_c].astype(np.float64) - rec_feat[:min_r, :min_c].astype(np.float64))
            feat_mae = float(np.mean(diff))
            feat_mse = float(np.mean(diff ** 2))
        else:
            feat_mae = float('nan')
            feat_mse = float('nan')
            
    # 3. Relations (Edges) Evaluation
    e_orig_set = set(orig["edges"])
    e_rec_set = set(rec["edges"])
    
    matched_edges = e_orig_set.intersection(e_rec_set)
    missing_edges = e_orig_set - e_rec_set
    extra_edges = e_rec_set - e_orig_set
    
    precision = len(matched_edges) / len(e_rec_set) if len(e_rec_set) > 0 else (1.0 if len(e_orig_set) == 0 else 0.0)
    recall = len(matched_edges) / len(e_orig_set) if len(e_orig_set) > 0 else (1.0 if len(e_rec_set) == 0 else 0.0)
    edge_f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # 4. Overall Semantic Fidelity Score
    semantic_fidelity = node_acc * edge_f1
    
    return {
        "node_count_orig": len(orig_nodes),
        "node_count_rec": len(rec_nodes),
        "node_acc": node_acc,
        "feat_mae": feat_mae,
        "feat_mse": feat_mse,
        "orig_edges_count": len(e_orig_set),
        "rec_edges_count": len(e_rec_set),
        "matched_edges_count": len(matched_edges),
        "missing_edges_count": len(missing_edges),
        "extra_edges_count": len(extra_edges),
        "missing_edges_list": sorted(list(missing_edges)),
        "extra_edges_list": sorted(list(extra_edges)),
        "edge_precision": precision,
        "edge_recall": recall,
        "edge_f1": edge_f1,
        "semantic_fidelity": semantic_fidelity
    }

## 7. Single Run Deep Inspection (`scene_data/scene_0000.bin` vs `received/received_scene_0000.bin`)

In [7]:
print("=" * 75)
print("DETAILED EVALUATION: DEFAULT ACTUAL VEINS RECEIVED FILE")
print("=" * 75)

bin_res = analyze_binary_integrity(SENT_BIN_PATH, ACTUAL_WIRELESS_RUNS["Good Channel (Actual Veins)"])
print(f"Sent Size            : {bin_res['size_sent']} bytes")
print(f"Received Size        : {bin_res['size_rec']} bytes")
print(f"Delivery Status      : {bin_res['delivery_status']}")
print(f"Sent SHA-256         : {bin_res['sent_sha']}")
print(f"Received SHA-256     : {bin_res['rec_sha']}")
print(f"Byte Identical       : {bin_res['is_identical']}")
print(f"Empirical BER        : {bin_res['ber']:.6e}")

sg_orig = decode_bin_file(SENT_BIN_PATH)
sg_rec = decode_bin_file(ACTUAL_WIRELESS_RUNS["Good Channel (Actual Veins)"])
sg_res = compare_scenegraphs(sg_orig, sg_rec)

print("-" * 75)
print(f"Nodes (Sent vs Rec)  : {sg_res['node_count_orig']} vs {sg_res['node_count_rec']}")
print(f"Node Label Accuracy  : {sg_res['node_acc'] * 100:.2f}%")
print(f"Feature MAE          : {sg_res['feat_mae']:.6e}")
print(f"Feature MSE          : {sg_res['feat_mse']:.6e}")
print(f"Edges (Sent vs Rec)  : {sg_res['orig_edges_count']} vs {sg_res['rec_edges_count']}")
print(f"Matched Edges        : {sg_res['matched_edges_count']}")
print(f"Missing Edges        : {sg_res['missing_edges_count']}")
print(f"Extra / False Edges  : {sg_res['extra_edges_count']}")
print(f"Relation Precision   : {sg_res['edge_precision'] * 100:.2f}%")
print(f"Relation Recall      : {sg_res['edge_recall'] * 100:.2f}%")
print(f"Relation F1-Score    : {sg_res['edge_f1'] * 100:.2f}%")
print(f"Semantic Fidelity    : {sg_res['semantic_fidelity'] * 100:.2f}%")
print("=" * 75)

DETAILED EVALUATION: DEFAULT ACTUAL VEINS RECEIVED FILE
Sent Size            : 862 bytes
Received Size        : 862 bytes
Delivery Status      : Delivered (Full Size)
Sent SHA-256         : 6cebe913d0be688c0b7732dadc44c7d30e2dfb06e84e20620d56b57603098d7c
Received SHA-256     : 6cebe913d0be688c0b7732dadc44c7d30e2dfb06e84e20620d56b57603098d7c
Byte Identical       : True
Empirical BER        : 0.000000e+00
---------------------------------------------------------------------------
Nodes (Sent vs Rec)  : 7 vs 7
Node Label Accuracy  : 100.00%
Feature MAE          : 0.000000e+00
Feature MSE          : 0.000000e+00
Edges (Sent vs Rec)  : 16 vs 16
Matched Edges        : 16
Missing Edges        : 0
Extra / False Edges  : 0
Relation Precision   : 100.00%
Relation Recall      : 100.00%
Relation F1-Score    : 100.00%
Semantic Fidelity    : 100.00%


## 8. Multi-Run Actual Wireless Outputs Benchmark Engine
Evaluates **all configured actual wireless outputs** (Good, Medium, Poor, etc.) from OMNeT++/Veins in batch mode and compiles a thesis-ready summary table.

In [8]:
def evaluate_actual_wireless_dataset(sent_path, wireless_runs_dict):
    """
    Iterates through all actual OMNeT++/Veins outputs and produces a comprehensive summary table.
    """
    sent_path = Path(sent_path)
    if not sent_path.is_file():
        raise FileNotFoundError(f"Ground truth sent file not found: {sent_path}")
        
    sg_orig = decode_bin_file(sent_path)
    records = []
    
    for condition_name, rec_path in wireless_runs_dict.items():
        rec_path = Path(rec_path)
        bin_info = analyze_binary_integrity(sent_path, rec_path)
        
        if not bin_info["file_exists"]:
            records.append({
                "Condition": condition_name,
                "Filename": rec_path.name,
                "Delivery Status": "File Pending / Missing",
                "Size (Bytes)": "N/A",
                "BER": float("nan"),
                "Decode Status": "N/A",
                "Node Accuracy (%)": float("nan"),
                "Feature MAE": float("nan"),
                "Feature MSE": float("nan"),
                "Relation Precision (%)": float("nan"),
                "Relation Recall (%)": float("nan"),
                "Relation F1 (%)": float("nan"),
                "Missing Edges": float("nan"),
                "Extra Edges": float("nan"),
                "Semantic Fidelity (%)": float("nan")
            })
            continue
            
        # Attempt Decoding
        try:
            sg_rec = decode_bin_file(rec_path)
            decode_status = "SUCCESS"
            comp = compare_scenegraphs(sg_orig, sg_rec)
            
            node_acc = comp["node_acc"] * 100
            feat_mae = comp["feat_mae"]
            feat_mse = comp["feat_mse"]
            rel_prec = comp["edge_precision"] * 100
            rel_rec = comp["edge_recall"] * 100
            rel_f1 = comp["edge_f1"] * 100
            missing_e = comp["missing_edges_count"]
            extra_e = comp["extra_edges_count"]
            sem_fid = comp["semantic_fidelity"] * 100
            
        except Exception as e:
            decode_status = f"FAILED ({type(e).__name__})"
            node_acc = 0.0
            feat_mae = float("nan")
            feat_mse = float("nan")
            rel_prec = 0.0
            rel_rec = 0.0
            rel_f1 = 0.0
            missing_e = len(sg_orig["edges"])
            extra_e = 0
            sem_fid = 0.0
            
        records.append({
            "Condition": condition_name,
            "Filename": rec_path.name,
            "Delivery Status": bin_info["delivery_status"],
            "Size (Bytes)": bin_info["size_rec"],
            "BER": bin_info["ber"],
            "Decode Status": decode_status,
            "Node Accuracy (%)": node_acc,
            "Feature MAE": feat_mae,
            "Feature MSE": feat_mse,
            "Relation Precision (%)": rel_prec,
            "Relation Recall (%)": rel_rec,
            "Relation F1 (%)": rel_f1,
            "Missing Edges": missing_e,
            "Extra Edges": extra_e,
            "Semantic Fidelity (%)": sem_fid
        })
        
    return pd.DataFrame(records)

# Execute multi-condition benchmark
benchmark_df = evaluate_actual_wireless_dataset(SENT_BIN_PATH, ACTUAL_WIRELESS_RUNS)
print("\n" + "=" * 85)
print("ACTUAL OMNeT++ / Veins WIRELESS CHANNELS EVALUATION TABLE")
print("=" * 85)
print(benchmark_df.to_string(index=False))



ACTUAL OMNeT++ / Veins WIRELESS CHANNELS EVALUATION TABLE
                    Condition                Filename        Delivery Status Size (Bytes)  BER Decode Status  Node Accuracy (%)  Feature MAE  Feature MSE  Relation Precision (%)  Relation Recall (%)  Relation F1 (%)  Missing Edges  Extra Edges  Semantic Fidelity (%)
  Good Channel (Actual Veins) received_scene_0000.bin  Delivered (Full Size)          862  0.0       SUCCESS              100.0          0.0          0.0                   100.0                100.0            100.0            0.0          0.0                  100.0
Medium Channel (Actual Veins)       received_test.bin  Delivered (Full Size)          862  0.0       SUCCESS              100.0          0.0          0.0                   100.0                100.0            100.0            0.0          0.0                  100.0
  Poor Channel (Actual Veins)       received_poor.bin File Pending / Missing          N/A  NaN           N/A                NaN          NaN

## 9. Comparative Visualization of Real Wireless Runs
Generates publication-ready comparative bar charts for actual wireless conditions (ignoring missing/pending files).

In [9]:
# Filter only successfully evaluated runs
valid_df = benchmark_df[benchmark_df["Decode Status"] == "SUCCESS"].copy()

if len(valid_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    
    # 1. Semantic Fidelity
    bars1 = axes[0].bar(valid_df["Condition"], valid_df["Semantic Fidelity (%)"], color="#2ca02c", width=0.4)
    axes[0].set_title("Semantic Fidelity (%)", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Fidelity (%)")
    axes[0].set_ylim(0, 110)
    axes[0].grid(axis="y", linestyle="--", alpha=0.6)
    axes[0].tick_params(axis='x', rotation=15)
    for bar in bars1:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{bar.get_height():.1f}%", ha='center', va='bottom')
        
    # 2. Node Accuracy vs Relation F1
    x = np.arange(len(valid_df))
    w = 0.3
    axes[1].bar(x - w/2, valid_df["Node Accuracy (%)"], width=w, label="Node Accuracy (%)", color="#1f77b4")
    axes[1].bar(x + w/2, valid_df["Relation F1 (%)"], width=w, label="Relation F1 (%)", color="#ff7f0e")
    axes[1].set_title("Node vs Relation Accuracy (%)", fontsize=12, fontweight="bold")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(valid_df["Condition"], rotation=15)
    axes[1].set_ylim(0, 110)
    axes[1].legend(loc="lower right")
    axes[1].grid(axis="y", linestyle="--", alpha=0.6)
    
    # 3. Bit Error Rate (BER)
    bars3 = axes[2].bar(valid_df["Condition"], valid_df["BER"], color="#d62728", width=0.4)
    axes[2].set_title("Empirical Bit Error Rate (BER)", fontsize=12, fontweight="bold")
    axes[2].set_ylabel("BER")
    axes[2].grid(axis="y", linestyle="--", alpha=0.6)
    axes[2].tick_params(axis='x', rotation=15)
    for bar in bars3:
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1e-6, f"{bar.get_height():.2e}", ha='center', va='bottom')
        
    plt.tight_layout()
    plt.show()
else:
    print("No valid evaluated runs to plot.")